# Learn linear regression and decision trees

FoML lab notebook. Run this in [Google Colab](https://colab.research.google.com/):

1. **File → Upload notebook** and pick this file.
2. **File → Save a copy in Drive** so a disconnect does not wipe your work.
3. Click a cell and press **Shift+Enter**. Or **Runtime → Run all**.

Colab already has NumPy, matplotlib, scikit-learn, and PyTorch.

You will:

1. Fit a line $y = a_0 + a_1 x$ by least squares (NumPy, sklearn) and by gradient descent (PyTorch).
2. Build a decision tree with **entropy / information gain** on the play / go-out table, by hand and with sklearn.

Trees are **not** trained in PyTorch. A split is a discrete `if`, so there is no gradient to backprop. PyTorch is for differentiable models (linear regression, neural nets).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_squared_error,
    r2_score,
)
from sklearn.preprocessing import LabelEncoder

print("numpy", np.__version__, "torch", torch.__version__)


---

# Part A — Linear regression

A **regression** problem: predict a number. Here the model is a straight line

$$
\hat y = a_0 + a_1 x
$$

- $a_1$ is the **slope** (sklearn `coef_`)
- $a_0$ is the **intercept** (sklearn `intercept_`)

With several points, no line goes through all of them. **Least squares** picks the line that minimises the sum of squared residuals:

$$
L = \sum_i (y_i - \hat y_i)^2
$$

In matrix form, stack a column of ones next to $x$ (the **design matrix** $X$) and solve $X^\top X w = X^\top y$. That is the same thing sklearn's `LinearRegression` does.


## A1. Two points by hand

Two points determine a unique line:

$$
a_1 = \frac{y_2 - y_1}{x_2 - x_1}, \qquad a_0 = y_1 - a_1 x_1
$$


In [ ]:
x1, y1 = 1.0, 2.0
x2, y2 = 4.0, 8.0
a1 = (y2 - y1) / (x2 - x1)
a0 = y1 - a1 * x1
print(f"line through (1, 2) and (4, 8):  y = {a0:.3f} + {a1:.3f} x")
print("check:", a0 + a1 * x1, a0 + a1 * x2)


## A2. Many points — NumPy least squares and sklearn

Toy data: hours studied vs score. sklearn wants `X` as 2-D `(n_samples, n_features)` even if there is only one feature — that is what `reshape(-1, 1)` does.


In [ ]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y = np.array([1.5, 3.2, 4.1, 5.8, 6.9, 8.4, 9.1, 11.0], dtype=float)

# design matrix [1, x]  →  y ≈ a0 * 1 + a1 * x
X_design = np.column_stack([np.ones_like(x), x])
a0, a1 = np.linalg.lstsq(X_design, y, rcond=None)[0]
print(f"numpy   y = {a0:.3f} + {a1:.3f} x")

X = x.reshape(-1, 1)
lin = LinearRegression().fit(X, y)
print(f"sklearn y = {lin.intercept_:.3f} + {lin.coef_[0]:.3f} x")

y_hat = lin.predict(X)
print("MSE", round(mean_squared_error(y, y_hat), 4))
print("R^2", round(r2_score(y, y_hat), 4))
print("R^2 = 1 is a perfect fit; 0 means 'always predicting the mean' is just as good.")


In [ ]:
xs = np.linspace(x.min(), x.max(), 50)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].scatter(x, y, label="data")
axes[0].plot(xs, lin.predict(xs.reshape(-1, 1)), label="least-squares fit")
axes[0].set_xlabel("hours studied (x)")
axes[0].set_ylabel("score (y)")
axes[0].set_title("Linear regression")
axes[0].legend()

resid = y - y_hat
axes[1].scatter(x, resid)
axes[1].axhline(0, color="gray", linewidth=1)
axes[1].set_xlabel("hours studied (x)")
axes[1].set_ylabel("y − ŷ")
axes[1].set_title("Residuals (should look like noise)")

fig.tight_layout()
plt.show()


## A3. Same line, PyTorch (gradient descent)

`nn.Linear(1, 1)` is $\hat y = w x + b$ — the same model. Instead of solving the normal equation, we walk downhill on MSE:

$$
\theta \leftarrow \theta - \eta \nabla L(\theta)
$$

Each epoch:

1. **predict** `y_hat = model(x)`
2. **loss** `MSE(y_hat, y)`
3. `optimizer.zero_grad()` — clear old gradients
4. `loss.backward()` — compute $\nabla L$
5. `optimizer.step()` — take a step of size $\eta$ (learning rate)

If $\eta$ is too big, the loss explodes. If it is too small, training crawls. After enough epochs, $w$ and $b$ should sit on the sklearn line.


In [ ]:
torch.manual_seed(0)
xt = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
yt = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
opt = torch.optim.SGD(model.parameters(), lr=0.02)

history = []
for epoch in range(2000):
    y_pred = model(xt)
    loss = loss_fn(y_pred, yt)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 400 == 0 or epoch == 1999:
        history.append((epoch, loss.item()))

b = model.bias.item()
w = model.weight.item()
print(f"pytorch y = {b:.3f} + {w:.3f} x")
print(f"sklearn y = {lin.intercept_:.3f} + {lin.coef_[0]:.3f} x")
print("loss by epoch:", history)

plt.figure(figsize=(5, 3.5))
plt.scatter(x, y, label="data")
plt.plot(xs, lin.predict(xs.reshape(-1, 1)), label="sklearn / lstsq")
plt.plot(xs, b + w * xs, "--", label="pytorch SGD")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Same line, two solvers")
plt.legend()
plt.show()


---

# Part B — Decision trees

A **classification** problem: predict a label (here Yes / No: go out?).

A tree asks one question per node, e.g. “what is the weather?”. Each answer sends you down a branch. A **leaf** predicts the class.

How to choose the question? Pick the feature whose split **drops entropy the most**.

$$
H = -\sum_c p_c \log_2 p_c
$$

- 5 Yes / 5 No → $H = 1$ bit (maximum confusion)
- all Yes → $H = 0$ (pure leaf — stop splitting)

**Information gain** of a feature:

$$
\mathrm{Gain} = H(\text{parent}) - \sum_{\text{child } v} \frac{n_v}{n} H(\text{child } v)
$$

Kurhekar uses **entropy**, not Gini (`1 - Σ p²`). In sklearn that is `criterion="entropy"`.

We recurse until a leaf is pure, or we hit `max_depth` / `min_samples_split`. A deep tree on 10 rows will memorise the table — that is **overfitting**.


## B1. Play / go-out table

Ten rows. Features: weather, temperature, humidity, wind. Label: play (go out?).


In [ ]:
rows = [
    # weather, temperature, humidity, wind, play
    ["Sunny",  "Hot",  "High",   "Weak",   "No"],
    ["Cloudy", "Hot",  "High",   "Weak",   "Yes"],
    ["Sunny",  "Mild", "Normal", "Strong", "Yes"],
    ["Cloudy", "Mild", "High",   "Strong", "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
    ["Rainy",  "Cool", "Normal", "Strong", "No"],
    ["Rainy",  "Mild", "High",   "Weak",   "Yes"],
    ["Sunny",  "Hot",  "High",   "Strong", "No"],
    ["Cloudy", "Hot",  "Normal", "Weak",   "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
]
feature_names = ["weather", "temp", "humidity", "wind"]
X_cat = np.array([r[:4] for r in rows])
y_cat = np.array([r[4] for r in rows])

print("n =", len(y_cat), "  Yes =", (y_cat == "Yes").sum(), "  No =", (y_cat == "No").sum())
print()
print(f"{'weather':8} {'temp':6} {'humidity':8} {'wind':6} play")
for r in rows:
    print(f"{r[0]:8} {r[1]:6} {r[2]:8} {r[3]:6} {r[4]}")


## B2. Entropy and information gain — compute it yourself

This is the part to understand, not just call sklearn. Weather should win.


In [ ]:
def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    p = p[p > 0]
    h = float(-(p * np.log2(p)).sum())
    return 0.0 if h < 1e-12 else h


def information_gain(column, labels):
    h_parent = entropy(labels)
    n = len(labels)
    remainder = 0.0
    for value in np.unique(column):
        mask = column == value
        remainder += mask.sum() / n * entropy(labels[mask])
    return h_parent - remainder


print(f"H(parent) = {entropy(y_cat):.3f}   (5 Yes / 5 No → 1 bit)\n")
gains = {}
for i, name in enumerate(feature_names):
    print(name)
    h_parent = entropy(y_cat)
    n = len(y_cat)
    remainder = 0.0
    for value in np.unique(X_cat[:, i]):
        mask = X_cat[:, i] == value
        h_child = entropy(y_cat[mask])
        remainder += mask.sum() / n * h_child
        print(
            f"    {value:8}  n={mask.sum():.0f}  "
            f"Yes={(y_cat[mask] == 'Yes').sum()}  No={(y_cat[mask] == 'No').sum()}  "
            f"H={h_child:.3f}"
        )
    gains[name] = h_parent - remainder
    print(f"  gain = {h_parent:.3f} − {remainder:.3f} = {gains[name]:.3f}\n")

best = max(gains, key=gains.get)
print("root split:", best, f"(gain {gains[best]:.3f})")
print("Cloudy is already a pure Yes leaf, so that branch stops.")


## B2b. Grow the ID3 tree (multiway categorical splits)

This is the tree the lecture draws: at each node pick the unused feature with highest gain, then branch **once per value** (Cloudy / Rainy / Sunny), not a numeric `<=` cut.

On the Sunny branch, **humidity and temperature have the same gain**. Both splits make pure leaves. The code breaks that tie the lecture way: Humidity, not Temp.

sklearn (next section) uses CART binary splits on encoded integers, so its picture can look different even when train accuracy is also 100%.


In [ ]:
PREFERRED = ["weather", "humidity", "wind", "temp"]  # lecture tie-break


def majority(labels):
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


def pick_split(names, gains):
    # highest gain; if tied, lecture order (humidity before temp on the Sunny branch)
    return min(
        range(len(names)),
        key=lambda k: (
            -gains[k],
            PREFERRED.index(names[k]) if names[k] in PREFERRED else 99,
        ),
    )


def build_id3(X, y, names):
    if len(np.unique(y)) == 1 or X.shape[1] == 0:
        return majority(y)
    gains = [information_gain(X[:, j], y) for j in range(X.shape[1])]
    j = pick_split(names, gains)
    feat = names[j]
    node = {"feature": feat, "gain": gains[j], "gains": dict(zip(names, gains)), "children": {}}
    for value in np.unique(X[:, j]):
        mask = X[:, j] == value
        keep = [k for k in range(X.shape[1]) if k != j]
        node["children"][str(value)] = build_id3(
            X[mask][:, keep], y[mask], [names[k] for k in keep]
        )
    return node


def print_id3(node, indent=0):
    pad = "  " * indent
    if not isinstance(node, dict):
        print(f"{pad}→ {node}")
        return
    extra = "  ".join(f"{n}={g:.3f}" for n, g in node["gains"].items())
    print(f"{pad}{node['feature']} (gain {node['gain']:.3f}; {extra})")
    for value, child in node["children"].items():
        print(f"{pad}  {node['feature']} = {value}")
        print_id3(child, indent + 2)


def predict_id3(node, sample):
    if not isinstance(node, dict):
        return node
    return predict_id3(node["children"][sample[node["feature"]]], sample)


id3_tree = build_id3(X_cat, y_cat, feature_names)
print_id3(id3_tree)

train_row = {"weather": "Sunny", "temp": "Hot", "humidity": "High", "wind": "Weak"}
unseen = {"weather": "Sunny", "temp": "Mild", "humidity": "High", "wind": "Weak"}
print()
print("train row", train_row, "→", predict_id3(id3_tree, train_row), "(table says No)")
print("unseen   ", unseen, "→", predict_id3(id3_tree, unseen), "(Sunny → Humidity=High → No)")
print()
print("On the Sunny branch, humidity and temp both have gain 0.918.")
print("The lecture tree draws Humidity; that is the tie-break above.")


## B3. sklearn tree (entropy)

`LabelEncoder` turns Sunny / Cloudy / Rainy into 0 / 1 / 2 so sklearn sees numbers. CART then makes **binary** cuts like `weather <= 0.5` (Cloudy vs the rest). That is why the plot may not look like the ID3 tree above, even though both fit these 10 rows perfectly.

Train accuracy 100% on n=10 is not skill — the tree can memorise the table.


In [ ]:
encoders = {
    name: LabelEncoder().fit(X_cat[:, i]) for i, name in enumerate(feature_names)
}
X_num = np.column_stack(
    [encoders[name].transform(X_cat[:, i]) for i, name in enumerate(feature_names)]
)
y_enc = LabelEncoder().fit(y_cat)
y_num = y_enc.transform(y_cat)
class_names = [str(c) for c in y_enc.classes_]

tree = DecisionTreeClassifier(criterion="entropy", random_state=0)
tree.fit(X_num, y_num)
pred = tree.predict(X_num)

print("classes", class_names)
for name in feature_names:
    print(name, "→", [str(c) for c in encoders[name].classes_])
print("train accuracy (n=10)", accuracy_score(y_num, pred))
print("confusion matrix (rows true No/Yes, cols pred No/Yes)")
print(confusion_matrix(y_num, pred, labels=y_enc.transform(["No", "Yes"])))
print()
print("100% on 10 training rows is not skill — the tree can memorise the table.")

plt.figure(figsize=(11, 6))
plot_tree(tree, feature_names=feature_names, class_names=class_names, filled=True)
plt.title("sklearn CART tree (entropy, binary splits)")
plt.show()


In [ ]:
def encode_day(day):
    return np.array(
        [[encoders[name].transform([day[name]])[0] for name in feature_names]]
    )


def sklearn_pred(day):
    return y_enc.inverse_transform(tree.predict(encode_day(day)))[0]


train_row = {"weather": "Sunny", "temp": "Hot", "humidity": "High", "wind": "Weak"}
unseen = {"weather": "Sunny", "temp": "Mild", "humidity": "High", "wind": "Weak"}

print("train row", train_row)
print("  ID3     ", predict_id3(id3_tree, train_row))
print("  sklearn ", sklearn_pred(train_row), "  (both match the table: No)")
print()
print("unseen combination", unseen)
print("  ID3     ", predict_id3(id3_tree, unseen), "  (Sunny → Humidity=High → No)")
print("  sklearn ", sklearn_pred(unseen), "  (binary CART can pick a different leaf)")
print()
print("Same 100% train accuracy, different answer on a row the table never showed.")
print("That is why you hold out a test set instead of trusting train accuracy.")


## B4. Why not PyTorch for trees?

Gradient descent needs $\nabla L$. A tree split is `if weather == Cloudy`. That `if` is not differentiable, so you cannot `loss.backward()` through it.

Use sklearn (or ID3 / C4.5 / CART) for trees. Use PyTorch for linear models and neural nets.


---

# Recap

| | Linear regression | Decision tree |
|---|---|---|
| Predicts | a number | a class |
| Model | $\hat y = a_0 + a_1 x$ | nested ifs |
| Fit by | least squares, or GD in PyTorch | greedy information gain |
| sklearn | `LinearRegression` → `coef_`, `intercept_` | `DecisionTreeClassifier(criterion="entropy")` |
| Metric here | MSE, $R^2$ | accuracy + confusion matrix |
| Trap | more features than independent rows | train accuracy 100% on tiny $n$ |

Next habit: **train / validation / test**. Fit on train, pick depth or learning rate on validation, report the test number **once**.
